# Medumba Two-Adapter + RAG Fallback (Colab)

This notebook trains two separate LoRA adapters and adds retrieval fallback:
- Adapter A: **French → Medumba**
- Adapter B: **Medumba → French**
- Retrieval fallback over your aligned pairs for grounded suggestions

In [ ]:
import os, json, platform, torch
print('Python:', platform.python_version())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip -q install -U transformers datasets peft accelerate sacrebleu sentencepiece safetensors rapidfuzz

## Data input
Upload or mount these files:
- `pairs_train.jsonl`
- `pairs_val.jsonl`
- `pairs_test.jsonl`

In [ ]:
# Option A: manual upload
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Option B: Google Drive (uncomment if preferred)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/MedumbaAI/outputs'
# TRAIN_FILE = f'{DATA_DIR}/pairs_train.jsonl'
# VAL_FILE = f'{DATA_DIR}/pairs_val.jsonl'
# TEST_FILE = f'{DATA_DIR}/pairs_test.jsonl'

TRAIN_FILE = '/content/pairs_train.jsonl'
VAL_FILE = '/content/pairs_val.jsonl'
TEST_FILE = '/content/pairs_test.jsonl'

for p in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    print(p, 'exists:', os.path.exists(p))

In [ ]:
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
)
from rapidfuzz import process, fuzz
import numpy as np

BASE_MODEL = 'google/byt5-small'
ROOT_OUT = '/content/medumba_two_adapters'
FR2MED_OUT = f'{ROOT_OUT}/fr2med_adapter'
MED2FR_OUT = f'{ROOT_OUT}/med2fr_adapter'

EPOCHS = 12
BATCH_SIZE = 8
GRAD_ACCUM = 2
LR = 5e-5
MAX_SOURCE_LEN = 128
MAX_TARGET_LEN = 128

os.makedirs(ROOT_OUT, exist_ok=True)

def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows_all = read_jsonl(TRAIN_FILE)
val_rows_all = read_jsonl(VAL_FILE)
test_rows_all = read_jsonl(TEST_FILE)

print('Train total:', len(train_rows_all), 'Val total:', len(val_rows_all), 'Test total:', len(test_rows_all))

In [ ]:
def train_direction(task_name, output_dir):
    print(f'\n=== Training {task_name} ===')
    train_rows = [r for r in train_rows_all if r.get('task') == task_name]
    val_rows = [r for r in val_rows_all if r.get('task') == task_name]

    print('Train rows:', len(train_rows), 'Val rows:', len(val_rows))
    if len(train_rows) == 0:
        raise ValueError(f'No train rows for task={task_name}')

    train_ds = Dataset.from_list(train_rows)
    val_ds = Dataset.from_list(val_rows)

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, use_safetensors=True)

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=16, lora_alpha=32, lora_dropout=0.1,
        target_modules=['q', 'v']
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    def tok(batch):
        x = tokenizer(batch['input'], truncation=True, max_length=MAX_SOURCE_LEN)
        y = tokenizer(text_target=batch['output'], truncation=True, max_length=MAX_TARGET_LEN)
        x['labels'] = y['input_ids']
        return x

    t_train = train_ds.map(tok, batched=True, remove_columns=train_ds.column_names)
    t_val = val_ds.map(tok, batched=True, remove_columns=val_ds.column_names)

    pad_id = tokenizer.pad_token_id
    valid_ratio = np.mean([any(tok != pad_id for tok in r['labels']) for r in t_train])
    print('Valid label ratio:', float(valid_ratio))

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        eval_strategy='epoch',
        save_strategy='epoch',
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        warmup_steps=100,
        lr_scheduler_type='cosine',
        max_grad_norm=1.0,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        logging_steps=25,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        fp16=False,
        bf16=use_bf16,
        report_to='none',
        remove_unused_columns=False
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=t_train,
        eval_dataset=t_val,
        data_collator=data_collator
    )

    trainer.train()
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f'Saved {task_name} adapter to {output_dir}')

train_direction('fr_to_medumba', FR2MED_OUT)
train_direction('medumba_to_fr', MED2FR_OUT)

In [ ]:
# Load base model + adapters for inference
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model_fr2med = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, use_safetensors=True)
base_model_med2fr = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, use_safetensors=True)

model_fr2med = PeftModel.from_pretrained(base_model_fr2med, FR2MED_OUT)
model_med2fr = PeftModel.from_pretrained(base_model_med2fr, MED2FR_OUT)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_fr2med.to(device).eval()
model_med2fr.to(device).eval()

def model_generate(text, direction):
    model = model_fr2med if direction == 'fr2med' else model_med2fr
    enc = base_tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=64, num_beams=4, do_sample=False)
    pred = base_tokenizer.decode(out[0], skip_special_tokens=True)
    return pred.strip()

In [ ]:
# Build retrieval stores from aligned pairs
fr2med_pairs = [r for r in (train_rows_all + val_rows_all + test_rows_all) if r.get('task') == 'fr_to_medumba']
med2fr_pairs = [r for r in (train_rows_all + val_rows_all + test_rows_all) if r.get('task') == 'medumba_to_fr']

# Deduplicate by input-output
def dedupe(rows):
    seen = set()
    out = []
    for r in rows:
        key = (r.get('input','').strip(), r.get('output','').strip())
        if key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out

fr2med_pairs = dedupe(fr2med_pairs)
med2fr_pairs = dedupe(med2fr_pairs)

fr2med_inputs = [r['input'] for r in fr2med_pairs]
med2fr_inputs = [r['input'] for r in med2fr_pairs]

def rag_candidates(text, direction, topk=3):
    if direction == 'fr2med':
        candidates = process.extract(text, fr2med_inputs, scorer=fuzz.WRatio, limit=topk)
        rows = fr2med_pairs
        inputs = fr2med_inputs
    else:
        candidates = process.extract(text, med2fr_inputs, scorer=fuzz.WRatio, limit=topk)
        rows = med2fr_pairs
        inputs = med2fr_inputs

    out = []
    for matched_text, score, _ in candidates:
        idx = inputs.index(matched_text)
        r = rows[idx]
        out.append({
            'score': float(score),
            'input': r['input'],
            'output': r['output'],
            'source': r.get('source', '')
        })
    return out

In [ ]:
# Hybrid translator: model first, fallback to retrieval when output looks weak
def weak_output(pred):
    if not pred:
        return True
    toks = pred.split()
    if len(pred) < 4:
        return True
    if len(toks) >= 2 and len(set(toks)) <= 1:
        return True
    return False

def translate_hybrid(text, direction='fr2med', topk=3):
    pred = model_generate(text, direction)
    cands = rag_candidates(text, direction, topk=topk)

    use_fallback = weak_output(pred)
    if use_fallback and cands:
        final = cands[0]['output']
        mode = 'fallback_rag'
    else:
        final = pred
        mode = 'model'

    return {
        'input': text,
        'direction': direction,
        'model_output': pred,
        'final_output': final,
        'mode': mode,
        'rag_candidates': cands
    }

In [ ]:
# Demo prompts (both directions)
demo_cases = [
    ('Bonjour mon ami', 'fr2med'),
    ('Je viens bientôt', 'fr2med'),
    ('Ma mère vient darriver', 'fr2med'),
    ('Ndàʼndàʼa sê’', 'med2fr'),
    ('Mə̂ α yǒg ne sə̂’ ə', 'med2fr')
]

for txt, d in demo_cases:
    r = translate_hybrid(txt, d, topk=3)
    print('='*70)
    print('INPUT      :', r['input'])
    print('DIRECTION  :', r['direction'])
    print('MODE       :', r['mode'])
    print('MODEL OUT  :', r['model_output'])
    print('FINAL OUT  :', r['final_output'])
    print('TOP RAG    :', r['rag_candidates'][:2])

In [ ]:
# Quick automatic evaluation for each direction
import sacrebleu

def eval_direction(task_name, direction, n=150):
    rows = [r for r in test_rows_all if r.get('task') == task_name][:n]
    preds = []
    refs = []
    em = 0
    for r in rows:
        out = translate_hybrid(r['input'], direction, topk=3)
        pred = out['final_output']
        ref = r['output']
        preds.append(pred)
        refs.append(ref)
        em += int(pred.strip() == ref.strip())
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score if rows else 0.0
    return {'rows': len(rows), 'exact_match': em/max(len(rows),1), 'chrf': chrf}

print('FR→MED:', eval_direction('fr_to_medumba', 'fr2med', n=150))
print('MED→FR:', eval_direction('medumba_to_fr', 'med2fr', n=150))

In [ ]:
# Export artifacts
import shutil
os.makedirs(ROOT_OUT, exist_ok=True)
archive_base = '/content/medumba_two_adapters_rag_demo'
archive_path = archive_base + '.zip'
if os.path.exists(archive_path):
    os.remove(archive_path)
shutil.make_archive(archive_base, 'zip', ROOT_OUT)
print('Archive created:', archive_path)

from google.colab import files
files.download(archive_path)

## Product notes
- Keep two adapters in production: `fr2med_adapter/` and `med2fr_adapter/`.
- Route by user choice of direction.
- If model output is weak, return top retrieval candidate + show source match for transparency.
- This architecture is the best bridge from tiny corpus to a future Medumba LLM roadmap.